In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RAIZ = Path("..") if Path("../data").exists() else Path(".")
df = pd.read_csv(RAIZ / "data" / "clean" / "dataset.csv")
df["hit"] = df["popularity_category"] == "High"

In [ ]:
def kpis(d):
    streams = d["stream_count"].sort_values(ascending=False)
    top1 = max(1, int(len(streams) * 0.01))
    return pd.Series({
        "músicas": len(d),
        "taxa de hits (%)": round(100 * d["hit"].mean(), 1),
        "streams da música típica (mediana)": d["stream_count"].median(),
        "streams no 1% mais ouvido (%)": round(100 * streams.iloc[:top1].sum() / streams.sum(), 1),
    })


kpis(df)

In [ ]:
por_genero = pd.DataFrame({genero: kpis(d) for genero, d in df.groupby("genre")}).T
display(por_genero.sort_values("taxa de hits (%)", ascending=False))

ax = por_genero["taxa de hits (%)"].sort_values().plot(kind="barh", color="#7F77DD", figsize=(7, 5))
ax.axvline(kpis(df)["taxa de hits (%)"], color="gray", linestyle="--", label="catálogo inteiro")
ax.set_xlabel("% das músicas com popularidade alta")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
faixas_pop = pd.cut(df["popularity"], bins=range(0, 101, 5), include_lowest=True)
streams_por_pop = df.groupby(faixas_pop, observed=True)["stream_count"].median()
display(streams_por_pop)

ax = streams_por_pop.plot(marker="o", color="#7F77DD", logy=True, figsize=(9, 4), rot=45)
ax.set_xlabel("popularidade (de 5 em 5)")
ax.set_ylabel("streams medianos (escala log)")
plt.tight_layout()
plt.show()

In [ ]:
peso = df.groupby("popularity_category").agg(musicas=("stream_count", "size"), streams=("stream_count", "sum"))
peso = (100 * peso / peso.sum()).round(1).reindex(["Low", "Medium", "High"])
display(peso)

ax = peso.T.plot(kind="barh", stacked=True, color=["#3AA59D", "#888780", "#7F77DD"], figsize=(8, 3))
ax.set_xlabel("%")
plt.tight_layout()
plt.show()